In [ ]:
# CELULA 1 - Configuração de Ambiente, GCP Client e Variáveis Globais

import os
import io
import zipfile
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client GCP
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Globais Dinâmicos
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"
BUCKET_SILVER = "cvm-lakehouse-silver-dev"

HOJE = datetime.now().date()
D1_LIMITE = HOJE - timedelta(days=1)
DATA_PISO_BACKFILL = datetime.strptime("2026-01-01", "%Y-%m-%d").date()

bucket_bronze = client.bucket(BUCKET_BRONZE)
bucket_silver = client.bucket(BUCKET_SILVER)

pd.set_option("display.max_columns", None)

# 4. Função Auxiliar Global: Limpeza e Padronização Absoluta de CNPJ
def limpar_cnpj(serie: pd.Series) -> pd.Series:
    """Remove caracteres não numéricos, resolve floats truncados (.0) e aplica zfill(14)."""
    return (
        serie.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(14)
    )

print("✔ Client GCP e utilitários inicializados para a CAMADA SILVER!")
print(f"✔ Source Bronze: gs://{BUCKET_BRONZE}")
print(f"✔ Target Silver: gs://{BUCKET_SILVER}")
print(f"✔ Data Limite D-1: {D1_LIMITE.strftime('%d/%m/%Y')}")

In [ ]:
# CELULA 2 - Carga e Consolidação do Universo de Ativos (CDA_FI CVM)

def carregar_carteiras_bronze(bucket) -> pd.DataFrame:
    """Varre o bucket Bronze e consolida todas as posições CDA em um único DataFrame."""
    print("=== [CARGA AMPLA] EXPANDINDO UNIVERSO DE ATIVOS DO CDA_FI ===")
    
    blobs_cda = [
        b for b in bucket.list_blobs(prefix="cvm/cda_fi/") 
        if b.name.endswith(".csv") or b.name.endswith(".zip")
    ]
    
    dfs_cda = []
    for blob in blobs_cda:
        buffer_cda = io.BytesIO()
        blob.download_to_file(buffer_cda)
        buffer_cda.seek(0)
        
        if blob.name.endswith(".zip"):
            with zipfile.ZipFile(buffer_cda) as z:
                for filename in z.namelist():
                    if filename.endswith(".csv"):
                        df_tmp = pd.read_csv(z.open(filename), sep=";", encoding="ISO-8859-1", low_memory=False)
                        dfs_cda.append(df_tmp)
        else:
            df_tmp = pd.read_csv(buffer_cda, sep=";", encoding="ISO-8859-1", low_memory=False)
            dfs_cda.append(df_tmp)

    if not dfs_cda:
        raise FileNotFoundError("Nenhum arquivo de carteira foi encontrado na Bronze.")

    df_amplo = pd.concat(dfs_cda, ignore_index=True)
    df_amplo.rename(columns=str.upper, inplace=True)

    col_cnpj = "CNPJ_FUNDO_CLASSE" if "CNPJ_FUNDO_CLASSE" in df_amplo.columns else "CNPJ_FUNDO"
    df_amplo["CNPJ_CLEAN"] = limpar_cnpj(df_amplo[col_cnpj])
    
    return df_amplo

# Execução da carga de ativos
try:
    df_ativos_detalhado = carregar_carteiras_bronze(bucket_bronze)
    print(f"✔ Sucesso! Posições de Ativos Carregadas : {len(df_ativos_detalhado):,}")
    print(f"✔ Universo de CNPJs Únicos de Ativos     : {df_ativos_detalhado['CNPJ_CLEAN'].nunique():,}")
except Exception as e:
    print(f"⚠️ Erro ao carregar a base de ativos da Bronze: {e}")

In [ ]:
# CELULA 3 - Carga da Planilha Referência do Quantum

def carregar_planilha_quantum(caminho_excel: str) -> pd.DataFrame:
    """Carrega e padroniza a planilha comercial exportada do Quantum Axis."""
    COLUNAS_QUANTUM = [
        'NOME_FUNDO', 'CNPJ_FUNDO', 'CNPJ_GESTAO', 'GESTORA', 
        'PCT_CDI_DIA', 'CAPTACAO_DIA', 'CAPTACAO_LIQUIDA_DIA', 
        'CAPTACAO_LIQUIDA_MES', 'CAPTACAO_LIQUIDA_ANO', 'CAPTACAO_LIQUIDA_30D', 
        'CAPTACAO_LIQUIDA_3M', 'CAPTACAO_LIQUIDA_6M', 'CAPTACAO_LIQUIDA_12M', 
        'CAPTACAO_LIQUIDA_24M', 'CAPTACAO_LIQUIDA_36M', 'VALOR_COTA', 
        'PATRIMONIO_LIQUIDO', 'RESGATE_DIA', 'RETORNO_DIA', 'TAXA_ADMINISTRACAO', 
        'CAPTACAO_LIQUIDA_M3', 'CAPTACAO_LIQUIDA_M2', 'CAPTACAO_LIQUIDA_M1', 
        'CAPTACAO_LIQUIDA_JANELA'
    ]
    
    df_q = pd.read_excel(
        caminho_excel, 
        skiprows=3, 
        names=COLUNAS_QUANTUM
    )
    df_q['CNPJ_CLEAN'] = limpar_cnpj(df_q['CNPJ_FUNDO'].dropna())
    return df_q

# Execução da carga do Quantum
CAMINHO_QUANTUM = r"C:\Users\victo\Downloads\captacao_quantum(2026-10-13).xlsx"

try:
    df_quantum_oficial = carregar_planilha_quantum(CAMINHO_QUANTUM)
    cnpjs_quantum_set = set(df_quantum_oficial['CNPJ_CLEAN'])
    print(f"✔ Planilha Quantum carregada com sucesso!")
    print(f"✔ Fundos Únicos no Quantum: {len(cnpjs_quantum_set):,}")
except Exception as e:
    df_quantum_oficial = None
    cnpjs_quantum_set = set()
    print(f"⚠️ Não foi possível carregar a planilha do Quantum local: {e}")

In [ ]:
# CELULA 4 - Preparação e Deduplicação da Carteira Recente por Fundo

def isolar_carteira_recente(df_ativos: pd.DataFrame) -> pd.DataFrame:
    """Trata valores de mercado e isola apenas a posição da última data de competência reportada por CNPJ."""
    print("⏳ Isolando a carteira mais recente de cada fundo...")
    
    df_temp = df_ativos.copy()
    
    # 1. Trata valores numéricos da posição
    df_temp['VL_MERC_POS_FINAL'] = pd.to_numeric(
        df_temp['VL_MERC_POS_FINAL'].astype(str).str.replace(',', '.'), 
        errors='coerce'
    ).fillna(0)

    # 2. Identifica e filtra a competência/posição máxima por fundo
    col_data = 'DT_COMPTC' if 'DT_COMPTC' in df_temp.columns else 'DT_POSICAO'
    max_datas = df_temp.groupby('CNPJ_CLEAN')[col_data].max().reset_index()

    df_recente = df_temp.merge(
        max_datas, 
        on=['CNPJ_CLEAN', col_data], 
        how='inner'
    )
    return df_recente

# Execução da deduplicação
if 'df_ativos_detalhado' in globals():
    df_carteira_recente = isolar_carteira_recente(df_ativos_detalhado)
    print(f"✔ Posições isoladas da carteira recente : {len(df_carteira_recente):,}")
    print(f"✔ CNPJs Únicos com carteira vigente      : {df_carteira_recente['CNPJ_CLEAN'].nunique():,}")
else:
    print("❌ 'df_ativos_detalhado' não encontrado na memória.")

In [ ]:
# CELULA 5 - Execução da Etapa 1 do Filtro (Crédito Privado e FICs)

def processar_etapa_1(df_carteira: pd.DataFrame, cnpjs_quantum: set = None) -> pd.DataFrame:
    """Aplica o filtro da Etapa 1: Pelo menos 10% de Crédito Direto OU ser FIC (Cotas >= 50%)."""
    print("=== EXECUTANDO ETAPA 1 (CRÉDITO PRIVADO & FICS) ===")
    
    PAPEIS_CREDITO_DIRETO = [
        'Debênture simples', 'Letra Financeira', 'CDB/ RDB', 'CRA', 
        'Certificado de recebíveis imobiliários', 'Nota Promissória/ Commercial Paper/ Export Note',
        'DPGE', 'Debênture Permutável', 'CDB Vinculado', 'Debênture conversível', 'FIDC', 'CCB'
    ]
    PAPEIS_COTAS = ['Fundo de Investimento e de Cotas', 'FI Participações']

    # 1. Agregação e alocação por fundo
    alocacao = df_carteira.groupby('CNPJ_CLEAN').agg(
        VL_CREDITO_DIRETO=('VL_MERC_POS_FINAL', lambda x: x[df_carteira.loc[x.index, 'TP_ATIVO'].isin(PAPEIS_CREDITO_DIRETO)].sum()),
        VL_COTAS=('VL_MERC_POS_FINAL', lambda x: x[df_carteira.loc[x.index, 'TP_ATIVO'].isin(PAPEIS_COTAS)].sum()),
        VL_TOTAL_CARTEIRA=('VL_MERC_POS_FINAL', 'sum')
    ).reset_index()

    df_entrante = alocacao[alocacao['VL_TOTAL_CARTEIRA'] > 0].copy()

    # 2. Cálculo dos percentuais de alocação
    df_entrante['PCT_CREDITO'] = df_entrante['VL_CREDITO_DIRETO'] / df_entrante['VL_TOTAL_CARTEIRA']
    df_entrante['PCT_COTAS'] = df_entrante['VL_COTAS'] / df_entrante['VL_TOTAL_CARTEIRA']
    df_entrante['TP_ESTRUTURA'] = np.where(df_entrante['PCT_COTAS'] >= 0.50, 'FIC', 'MASTER/DIRETO')

    # 3. Regra de aprovação
    cond_e1 = (df_entrante['PCT_CREDITO'] >= 0.10) | (df_entrante['PCT_COTAS'] >= 0.50)
    df_aprovados = df_entrante[cond_e1].copy()

    # 4. Auditoria e Relatório
    total_entrantes = len(df_entrante)
    total_aprovados = len(df_aprovados)
    total_removidos = total_entrantes - total_aprovados
    aum_entrante = df_entrante['VL_TOTAL_CARTEIRA'].sum()
    aum_aprovado = df_aprovados['VL_TOTAL_CARTEIRA'].sum()

    print("\n" + "=" * 80)
    print("📊 RESULTADO DA ETAPA 1 (CRÉDITO PRIVADO E FICS)")
    print("=" * 80)
    print(f"• Total Analisado na CVM 175  : {total_entrantes:,} veículos (AUM: R$ {aum_entrante/1e9:,.2f} Bi)")
    print(f"• Aprovados na Etapa 1        : {total_aprovados:,} veículos ({(total_aprovados/total_entrantes)*100:.1f}%) | AUM Retido: R$ {aum_aprovado/1e9:,.2f} Bi")
    print(f"• Filtrados                   : {total_removidos:,} veículos ({(total_removidos/total_entrantes)*100:.1f}%)")
    print("-" * 80)
    masters_cnt = (df_aprovados['TP_ESTRUTURA'] == 'MASTER/DIRETO').sum()
    fics_cnt = (df_aprovados['TP_ESTRUTURA'] == 'FIC').sum()
    print(f"  - Masters / Carteiras Diretas Crédito : {masters_cnt:,}")
    print(f"  - FICs / Veículos em Cotas            : {fics_cnt:,}")

    if cnpjs_quantum:
        matches = cnpjs_quantum.intersection(set(df_aprovados['CNPJ_CLEAN']))
        print("-" * 80)
        print(f"🎯 Retenção da Planilha Quantum: {len(matches):,} de {len(cnpjs_quantum):,} ({(len(matches)/len(cnpjs_quantum))*100:.1f}% Aderência)")
    print("=" * 80)

    return df_aprovados

# Execução da Etapa 1
if 'df_carteira_recente' in globals():
    df_e1_aprovados = processar_etapa_1(df_carteira_recente, cnpjs_quantum_set)
else:
    print("❌ 'df_carteira_recente' não encontrado na memória.")

In [ ]:
#geracao de arquivo consolidado para analise exploratoria junto aos gestores de fundos reais que podem auxiliar na delimitacao da amostragem

print("=== EXECUTANDO ETAPA 1 (CRÉDITO PRIVADO & FICS - DICIONÁRIO EXPANDIDO) ===")

# 1. Trata valores numéricos da posição
df_ativos_detalhado['VL_MERC_POS_FINAL'] = pd.to_numeric(
    df_ativos_detalhado['VL_MERC_POS_FINAL'].astype(str).str.replace(',', '.'), 
    errors='coerce'
).fillna(0)

# 2. ISOLAMENTO DEDUPLICADO (Máxima Data de Competência)
col_data = 'DT_COMPTC' if 'DT_COMPTC' in df_ativos_detalhado.columns else 'DT_POSICAO'
max_datas_por_cnpj = df_ativos_detalhado.groupby('CNPJ_CLEAN')[col_data].max().reset_index()

df_carteira_recente = df_ativos_detalhado.merge(
    max_datas_por_cnpj, 
    on=['CNPJ_CLEAN', col_data], 
    how='inner'
)

# 3. Mapeamento EXPANDIDO de Papéis de Crédito Direto e Cotas
PAPEIS_CREDITO_DIRETO = [
    'Debênture simples', 'Letra Financeira', 'CDB/ RDB', 'CRA', 
    'Certificado de recebíveis imobiliários', 'Nota Promissória/ Commercial Paper/ Export Note',
    'DPGE', 'Debênture Permutável', 'CDB Vinculado', 'Debênture conversível', 'FIDC', 'CCB',
    'Nota Comercial', 'NC', 'LIG', 'Letra Imobiliária Garantida', 'Debêntures', 'DEBÊNTURES',
    'Título Securitizado', 'Direitos Creditórios', 'Cédula de Crédito Bancário'
]

PAPEIS_COTAS = [
    'Fundo de Investimento e de Cotas', 'FI Participações', 'Fundo de Cotas',
    'FUNDO DE COTAS DE FUNDOS DE INVESTIMENTO', 'Cotas de Fundos de Investimento',
    'Fundo de Investimento Imobiliário'
]

# 4. Agregação por CNPJ sobre a Carteira Recente Deduplicada
alocacao_e1 = df_carteira_recente.groupby('CNPJ_CLEAN').agg(
    VL_CREDITO_DIRETO=('VL_MERC_POS_FINAL', lambda x: x[df_carteira_recente.loc[x.index, 'TP_ATIVO'].isin(PAPEIS_CREDITO_DIRETO)].sum()),
    VL_COTAS=('VL_MERC_POS_FINAL', lambda x: x[df_carteira_recente.loc[x.index, 'TP_ATIVO'].isin(PAPEIS_COTAS)].sum()),
    VL_TOTAL_CARTEIRA=('VL_MERC_POS_FINAL', 'sum')
).reset_index()

df_e1_entrante = alocacao_e1[alocacao_e1['VL_TOTAL_CARTEIRA'] > 0].copy()

# 5. Cálculo dos Percentuais
df_e1_entrante['PCT_CREDITO'] = df_e1_entrante['VL_CREDITO_DIRETO'] / df_e1_entrante['VL_TOTAL_CARTEIRA']
df_e1_entrante['PCT_COTAS'] = df_e1_entrante['VL_COTAS'] / df_e1_entrante['VL_TOTAL_CARTEIRA']
df_e1_entrante['TP_ESTRUTURA'] = np.where(df_e1_entrante['PCT_COTAS'] >= 0.50, 'FIC', 'MASTER/DIRETO')

# 6. Regra da Etapa 1 (10% Crédito OU 50% Cotas)
cond_e1_ok = (df_e1_entrante['PCT_CREDITO'] >= 0.10) | (df_e1_entrante['PCT_COTAS'] >= 0.50)
df_e1_aprovados = df_e1_entrante[cond_e1_ok].copy()

# 7. Reconciliação Integrada com o Quantum
cnpjs_e1_aprovados = set(df_e1_aprovados['CNPJ_CLEAN'])
if 'df_quantum_oficial' in globals():
    col_cnpj_q = 'CNPJ_CLEAN' if 'CNPJ_CLEAN' in df_quantum_oficial.columns else 'CNPJ_FUNDO'
    df_quantum_oficial['CNPJ_CLEAN'] = df_quantum_oficial[col_cnpj_q].astype(str).str.replace(r'\.0$', '', regex=True).str.replace(r'\D', '', regex=True).str.zfill(14)
    cnpjs_quantum = set(df_quantum_oficial['CNPJ_CLEAN'])
else:
    cnpjs_quantum = set()

matches_quantum = cnpjs_quantum.intersection(cnpjs_e1_aprovados)
quantum_perdidos = cnpjs_quantum - cnpjs_e1_aprovados

# 8. Relatório de Volumetria
total_entrantes = len(df_e1_entrante)
total_aprovados = len(df_e1_aprovados)

print("\n" + "=" * 80)
print("📊 RESULTADO DA ETAPA 1 (COM DICIONÁRIO AMPLIADO)")
print("=" * 80)
print(f"• Total Aprovado na CVM : {total_aprovados:,} veículos")

if cnpjs_quantum:
    pct_aderencia = (len(matches_quantum) / len(cnpjs_quantum)) * 100
    print("-" * 80)
    print(f"🎯 NOVA COBERTURA DO QUANTUM: {len(matches_quantum):,} de {len(cnpjs_quantum):,} ({pct_aderencia:.1f}% de Aderência)")
    print(f"⚠️ Fundos do Quantum ainda pendentes: {len(quantum_perdidos):,}")
print("=" * 80)

import pandas as pd
import os

print("=== CONSTRUINDO A BASE MASTER CVM 175 ===")

if 'alocacao_e1' in globals() and 'cadastro_ativo' in globals():
    
    # 1. Prepara a base de Cadastro (todas as colunas)
    df_cad_full = cadastro_ativo.copy()
    
    # Descobre dinamicamente qual é o nome da coluna de CNPJ no cadastro
    if 'CNPJ_CLEAN' in df_cad_full.columns:
        col_cnpj_cad = 'CNPJ_CLEAN'
    elif 'CNPJ_FUNDO_CLEAN' in df_cad_full.columns:
        col_cnpj_cad = 'CNPJ_FUNDO_CLEAN'
    elif 'CNPJ_FUNDO' in df_cad_full.columns:
        col_cnpj_cad = 'CNPJ_FUNDO'
    else:
        raise KeyError(f"Nenhuma coluna de CNPJ encontrada. Colunas disponíveis: {df_cad_full.columns.tolist()}")
    
    # Padroniza a chave de cruzamento no cadastro
    df_cad_full['CNPJ_CLEAN'] = (
        df_cad_full[col_cnpj_cad]
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.replace(r'\D', '', regex=True)
        .str.zfill(14)
    )
    
    # 2. Prepara a base de Carteiras Agregadas (que já rodamos na Etapa 1)
    df_carteira_agg = alocacao_e1.copy()
    
    # Recalcula percentuais para análise no Excel
    df_carteira_agg['PCT_CREDITO'] = df_carteira_agg['VL_CREDITO_DIRETO'] / df_carteira_agg['VL_TOTAL_CARTEIRA']
    df_carteira_agg['PCT_COTAS'] = df_carteira_agg['VL_COTAS'] / df_carteira_agg['VL_TOTAL_CARTEIRA']
    
    # 3. O Cruzamento Master (Carteira D-90 + Cadastro CVM Completo)
    # Mantém todos os fundos que têm carteira e anexa os dados cadastrais correspondentes
    df_cvm_master = df_carteira_agg.merge(df_cad_full, on='CNPJ_CLEAN', how='left')
    
    # 4. Organização das Colunas (Coloca as mais importantes primeiro)
    colunas_prioritarias = [
        'CNPJ_CLEAN', 'DENOM_SOCIAL', 'SIT', 'FUNDO_EXCLUSIVO', 'PUBLICO_ALVO',
        'VL_TOTAL_CARTEIRA', 'PCT_CREDITO', 'PCT_COTAS', 'VL_CREDITO_DIRETO', 'VL_COTAS'
    ]
    # Pega as colunas prioritárias que existem na base e adiciona todo o resto depois
    cols_existentes_prio = [c for c in colunas_prioritarias if c in df_cvm_master.columns]
    outras_cols = [c for c in df_cvm_master.columns if c not in cols_existentes_prio]
    
    df_cvm_master = df_cvm_master[cols_existentes_prio + outras_cols]
    
    # 5. Exportação para Excel/CSV
    nome_arquivo = "Base_Master_CVM_175_Gestores.csv"
    
    print(f"⏳ Exportando base com {len(df_cvm_master):,} fundos e {len(df_cvm_master.columns)} colunas...")
    
    # Salva em CSV com BOM (utf-8-sig) para que o Excel abra os acentos em português automaticamente e separador ';'
    df_cvm_master.to_csv(nome_arquivo, sep=';', index=False, encoding='utf-8-sig', decimal=',')
    
    print(f"✅ SUCESSO! Arquivo guardado no seu computador em:\n -> {os.path.join(os.getcwd(), nome_arquivo)}")
    print("\n💡 Dica para os Gestores: Abram o ficheiro no Excel, apliquem os filtros na coluna 'SIT', 'FUNDO_EXCLUSIVO' e 'VL_TOTAL_CARTEIRA', e definam as regras de corte que vamos aplicar no robô depois!")

else:
    print("❌ ERRO: As tabelas 'alocacao_e1' ou 'cadastro_ativo' não estão na memória.")